# LoRA fine-tuned LLM selector

Fine-tunes an instruction-tuned LLM with QLoRA to pick the gold candidate letter from TermMap
prompts, then scores request files with the fine-tuned model.
Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

All files are read from and written to `My Drive/HiLabs_Clinical`.

| RUN | Train on | Score | Outputs |
|---|---|---|---|
| `dev` | `llm_train_dev.jsonl` | `llm_requests_dev.jsonl` | `outputs/lora_scores_dev.jsonl` |
| `full` | `llm_train_full.jsonl` | `llm_requests_<name>.jsonl` | `outputs/lora_scores_<name>.jsonl` |

Training examples come from out-of-fold candidate lists. The gold code is inserted when retrieval
missed it and the candidate order is shuffled, so the model cannot rely on position. The loss is
computed on the answer letter only. Checkpoints are written to `cache/lora_<RUN>/` during training,
so a disconnected session resumes where it stopped.

In [ ]:
RUN = "dev"                      # "dev" or "full"
SCORE_NAMES = ["dev"]            # for RUN="full", e.g. ["eval"]
BASE = "Qwen/Qwen2.5-3B-Instruct" # "Qwen/Qwen2.5-7B-Instruct" if there is GPU time for it
MAX_TRAIN = None                 # cap on training examples, None for all
MAX_LEN = 1280
EPOCHS = 1
LR = 2e-4
SEED = 13

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes peft

In [ ]:
import glob, inspect, json, os, random, time, numpy as np, torch
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/HiLabs_Clinical"
assert os.path.isdir(WORK), f"create the folder {WORK} in Google Drive first"
CACHE, OUT = f"{WORK}/cache", f"{WORK}/outputs"
os.makedirs(CACHE, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer,
                          TrainingArguments)

assert torch.cuda.is_available(), "switch the runtime to a GPU"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
LETTERS = "ABCDEFGHIJ"
ADAPTER_DIR = f"{CACHE}/lora_{RUN}/adapter"
CKPT_DIR = f"{CACHE}/lora_{RUN}/checkpoints"

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
letter_ids = [tok.encode(L, add_special_tokens=False)[0] for L in LETTERS]
assert len(set(letter_ids)) == len(LETTERS)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True)

In [ ]:
class LetterData(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        r = self.items[i]
        prompt = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True)
        ids = tok(prompt, add_special_tokens=False)["input_ids"][-(MAX_LEN - 1):]
        ans = tok.encode(r["answer"], add_special_tokens=False)[0]
        return {"input_ids": ids + [ans], "labels": [-100] * len(ids) + [ans]}

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    return {
        "input_ids": torch.tensor([b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch]),
        "labels": torch.tensor([b["labels"] + [-100] * (n - len(b["labels"])) for b in batch]),
        "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (n - len(b["input_ids"])) for b in batch]),
    }

if not os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"):
    items = read_jsonl(f"{WORK}/llm_train_{RUN}.jsonl")
    random.Random(SEED).shuffle(items)
    if MAX_TRAIN:
        items = items[:MAX_TRAIN]
    print(len(items), "training examples")
    base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto",
                                                torch_dtype=torch.float16)
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    model = get_peft_model(base, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()
    total_steps = EPOCHS * -(-len(items) // (2 * 8))
    wanted = dict(
        output_dir=CKPT_DIR, per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.05 * total_steps)),
        fp16=True, logging_steps=20, save_steps=100, save_total_limit=2, seed=SEED, data_seed=SEED,
        report_to=[], remove_unused_columns=False, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}, optim="paged_adamw_8bit")
    # pass only the arguments this transformers version accepts
    accepted = inspect.signature(TrainingArguments.__init__).parameters
    dropped = sorted(k for k in wanted if k not in accepted)
    if dropped:
        print("not supported by this transformers version, skipped:", dropped)
    args = TrainingArguments(**{k: v for k, v in wanted.items() if k in accepted})
    trainer = Trainer(model=model, args=args, train_dataset=LetterData(items), data_collator=collate)
    resume = bool(glob.glob(f"{CKPT_DIR}/checkpoint-*"))
    t = time.time()
    trainer.train(resume_from_checkpoint=resume or None)
    print(f"trained in {time.time() - t:.0f}s")
    model.save_pretrained(ADAPTER_DIR)
    del trainer, model, base
    torch.cuda.empty_cache()
print("adapter:", ADAPTER_DIR)

In [ ]:
tok.padding_side = "left"
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()

@torch.no_grad()
def score(reqs, batch=8):
    prompts = [tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True) for r in reqs]
    order = sorted(range(len(reqs)), key=lambda i: len(prompts[i]))
    out = [None] * len(reqs)
    t = time.time()
    for b in range(0, len(order), batch):
        idx = order[b:b + batch]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        logits = model(**enc).logits[:, -1, :].float()
        for i, row in zip(idx, logits):
            n = len(reqs[i]["codes"])
            p = torch.softmax(row[letter_ids[:n]], dim=-1).cpu().tolist()
            out[i] = {"id": reqs[i]["id"], "probs": [round(x, 6) for x in p]}
        if (b // batch) % 50 == 0:
            print(f"{b + len(idx)}/{len(reqs)}  {time.time() - t:.0f}s", flush=True)
    return out

for name in SCORE_NAMES:
    reqs = read_jsonl(f"{WORK}/llm_requests_{name}.jsonl")
    scores = score(reqs)
    path = f"{OUT}/lora_scores_{name}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for s in scores:
            fh.write(json.dumps(s) + "\n")
    print(name, len(scores), "->", path)